In [1]:
import openai
import pandas as pd
import cohere

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

In [2]:
from dotenv import load_dotenv
load_dotenv("../../.env")

True

### Retrieval

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
def get_embedding(text, model = "text-embedding-3-small"):
    response = openai.embeddings.create(
        input =text, 
        model=model
    )
    return response.data[0].embedding  

In [5]:
def retrieve_data_weighted(query, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [8]:
results = retrieve_data_weighted("Can I get a tablet", k=20)

In [9]:
results

{'retrieved_context_ids': ['B0BN58Z4YX',
  'B09RHF4L45',
  'B09RN3KN5C',
  'B0BCFYCXRH',
  'B0BSD3QK7M',
  'B0B58CPFWY',
  'B0BHVH4D37',
  'B0B157WDDJ',
  'B0B159KDFP',
  'B0B6ZZH83Y',
  'B0BGHBL86V',
  'B0C7DCS2KW',
  'B09WR36NK8',
  'B0BT83RRJ2',
  'B07Y36DDYM',
  'B09XQMRYBJ',
  'B0BWRZ1HRD',
  'B09SZNLXJQ',
  'B0B2KJKT86',
  'B0BZCM9CBR'],
 'retrieved_context': ['ASWINN Tablet Tripod Stand, Gooseneck 65" Height Adjustable Tablet Stand Floor with 360° Rotating Tripod Mount Suitable for iPhone,Tablet,Kindle and All 4.5-12.9 Inch Tablet and Phone (Black) 【Free Your Hands】: With this gooseneck arm tablet tripod stand, you can find a more comfortable way to use tablet & Phone in bed or sofa in winter. Release your hands while make your leisure time more enjoyable，the best Christmas gift or New Year gifts for wife,husband,elders,friend or colleague. 【Versatile Gooseneck Tablet Tripod Stand】This ergonomic gooseneck tablet holder suits most tablets and phones. you can easily get your desir

### Reranking

In [6]:
cohere_client = cohere.ClientV2()

In [10]:
to_rerank = results["retrieved_context"]

In [11]:
to_rerank

['ASWINN Tablet Tripod Stand, Gooseneck 65" Height Adjustable Tablet Stand Floor with 360° Rotating Tripod Mount Suitable for iPhone,Tablet,Kindle and All 4.5-12.9 Inch Tablet and Phone (Black) 【Free Your Hands】: With this gooseneck arm tablet tripod stand, you can find a more comfortable way to use tablet & Phone in bed or sofa in winter. Release your hands while make your leisure time more enjoyable，the best Christmas gift or New Year gifts for wife,husband,elders,friend or colleague. 【Versatile Gooseneck Tablet Tripod Stand】This ergonomic gooseneck tablet holder suits most tablets and phones. you can easily get your desired angle to meet your needs at different angles and positions. Except for watching movies, you are also allowed to use your tablet for visual presentations, like reading sheet music, watching videos, zoom calls, online teaching, public speaking, reading e-books, giving performances, etc. 【Wide Compatibility】It suitable for various tablets and mobile devices ranging 

In [12]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query="Can I get a tablet?",
    documents=to_rerank,
    top_n=20,
)

In [13]:
response

V2RerankResponse(id='310d787c-22cf-4720-a51a-84aaa867290d', results=[V2RerankResponseResultsItem(index=1, relevance_score=0.8699799), V2RerankResponseResultsItem(index=2, relevance_score=0.8632072), V2RerankResponseResultsItem(index=6, relevance_score=0.8502698), V2RerankResponseResultsItem(index=4, relevance_score=0.84826964), V2RerankResponseResultsItem(index=12, relevance_score=0.84317327), V2RerankResponseResultsItem(index=3, relevance_score=0.8374073), V2RerankResponseResultsItem(index=8, relevance_score=0.83473045), V2RerankResponseResultsItem(index=7, relevance_score=0.8331075), V2RerankResponseResultsItem(index=11, relevance_score=0.8320184), V2RerankResponseResultsItem(index=16, relevance_score=0.8208144), V2RerankResponseResultsItem(index=10, relevance_score=0.80415994), V2RerankResponseResultsItem(index=0, relevance_score=0.77294225), V2RerankResponseResultsItem(index=5, relevance_score=0.7660137), V2RerankResponseResultsItem(index=9, relevance_score=0.7546258), V2RerankResp

In [14]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [15]:
reranked_results

['plimpton Android Tablet 10 inch - 8 Core Processor, 4GB RAM 64GB Storage, 1.8M Length USB-C Cable, Android 11, 1920 x 1200 FHD IPS, 13MP Sony Rear Camera, 6000mAh Battery, 2.4G+5G WiFi Tablets PC High Performance Tablet 10.1 inch: Features a faster 1.8GHz Octa-Core CPU and 4GB RAM, you can not only obtain seamless multitasking and hyper-fast content streaming experience, but also can view social media, watch videos, play games, complete homework or finish work. With a 64GB internal storage and support a SD card expanded to 512GB(NOT included), no longer worry about insufficient memory.★★Please note this is a Wi-Fi tablet, not compatible with SIM card 2022 Latest Android 11 Tablets: Equipped with the latest Android 11 system (GMS certified). Automatic subtitleing, smart reply, notification "bubble", dark theme, eye protection mode etc. It also launches apps 20% faster than Android 10 OS. New control spaces and more privacy features allow you to create a rich user central expression. Y